# Transformação da camada Silver para a camada Gold

**Tech Challenge Fase 3** · Pós-Tech em Data Analytics, FIAP
**Base:** State of Data Brazil, edições 2023-2024, 2024-2025 e 2025-2026
**Etapa do pipeline:** construção das tabelas analíticas

A camada Gold é organizada por pergunta de negócio. Cada tabela responde a um conjunto de perguntas do enunciado e já chega pronta para consulta no Athena e para geração de gráficos.

## 1. Configuração

In [1]:
import sys, os
os.environ.pop("JAVA_TOOL_OPTIONS", None)
sys.path.insert(0, "../src")

from pyspark.sql import SparkSession, functions as F

spark = (
    SparkSession.builder.appName("tc3")
    .master("local[2]")                      # no AWS Glue esta linha nao existe
    .config("spark.driver.memory", "3g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.sql.session.timeZone", "America/Sao_Paulo")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version)

26/09/05 21:53:51 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/09/05 21:53:51 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/05 21:53:51 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 3.5.3


## 2. Execução do Glue Job 02

In [2]:
from job02_silver_gold import tabela_distribuicoes, tabela_cruzamentos, tabela_salario, tabela_mencoes

silver = spark.read.parquet("../dados/silver").cache()
print(f"camada Silver: {silver.count()} linhas")

distribuicoes = tabela_distribuicoes(silver)
cruzamentos = tabela_cruzamentos(silver)
salario = tabela_salario(silver)
mencoes = tabela_mencoes(silver)

for nome, df in [("distribuicoes", distribuicoes), ("cruzamentos", cruzamentos),
                 ("salario", salario), ("mencoes", mencoes)]:
    print(f"gold_{nome}: {df.count()} linhas")

camada Silver: 14002 linhas


gold_distribuicoes: 1008 linhas


gold_cruzamentos: 982 linhas


gold_salario: 135 linhas


gold_mencoes: 29 linhas


## 3. Estrutura das tabelas

Todas são tabelas longas, formato que facilita tanto a consulta SQL quanto o gráfico.

In [3]:
distribuicoes.printSchema()
distribuicoes.filter("dimensao = 'genero'").orderBy("categoria", "edicao").show(12, truncate=False)

root
 |-- edicao: string (nullable = true)
 |-- pergunta_enunciado: string (nullable = false)
 |-- dimensao: string (nullable = false)
 |-- categoria: string (nullable = true)
 |-- respondentes: long (nullable = false)
 |-- total_validos: long (nullable = true)
 |-- participacao_pct: double (nullable = true)



+---------+------------------+--------+--------------------+------------+-------------+----------------+
|edicao   |pergunta_enunciado|dimensao|categoria           |respondentes|total_validos|participacao_pct|
+---------+------------------+--------+--------------------+------------+-------------+----------------+
|2023-2024|R19               |genero  |Feminino            |1293        |5293         |24.43           |
|2024-2025|R19               |genero  |Feminino            |1225        |5215         |23.49           |
|2025-2026|R19               |genero  |Feminino            |767         |3494         |21.95           |
|2023-2024|R19               |genero  |Masculino           |3975        |5293         |75.1            |
|2024-2025|R19               |genero  |Masculino           |3967        |5215         |76.07           |
|2025-2026|R19               |genero  |Masculino           |2707        |3494         |77.48           |
|2023-2024|R19               |genero  |Outro           

**Leitura do resultado.** A participação feminina cai de 24,4% para 22,0% ao longo da série. A série tem três pontos e, portanto, duas transições, e a queda ocorre nas duas, sempre na mesma direção, somando 2,5 pontos. É tendência, não oscilação de amostra.

## 4. Premissa da estatística salarial

A pesquisa coleta faixa de remuneração, não valor. Toda estatística usa o ponto médio da faixa e é apresentada como faixa mediana, nunca como salário médio. A faixa aberta do topo usa o próprio limite inferior, o que é conservador.

In [4]:
salario.filter("recorte = 'nivel'").orderBy("categoria", "edicao").show(truncate=False)

+---------+-------+-------------------+------------+-------------------+--------------+--------------+
|edicao   |recorte|categoria          |respondentes|mediana_faixa_reais|q1_faixa_reais|q3_faixa_reais|
+---------+-------+-------------------+------------+-------------------+--------------+--------------+
|2025-2026|nivel  |Especialista/Staff+|349         |18000              |14000         |22500         |
|2023-2024|nivel  |Júnior             |1046        |3500               |2500          |5000          |
|2024-2025|nivel  |Júnior             |868         |3500               |2500          |5000          |
|2025-2026|nivel  |Júnior             |518         |3500               |2500          |5000          |
|2023-2024|nivel  |Pleno              |1392        |7000               |5000          |10000         |
|2024-2025|nivel  |Pleno              |1376        |7000               |5000          |10000         |
|2025-2026|nivel  |Pleno              |775         |7000               |5

**Leitura do resultado.** A faixa mediana do nível Sênior subiu de R$ 10.000 para R$ 14.000 entre a primeira e a segunda edição, alta de 40%, e depois estabilizou. Júnior e Pleno ficaram parados nas três edições. Para uma empresa que precisa montar time, isso significa que contratar pronto ficou mais caro e formar não.

## 5. Gravação da camada Gold

In [5]:
for nome, df in [("gold_distribuicoes", distribuicoes), ("gold_cruzamentos", cruzamentos),
                 ("gold_salario", salario), ("gold_mencoes", mencoes)]:
    df.coalesce(1).write.mode("overwrite").parquet(f"../dados/gold/{nome}")
    print(f"gravada: {nome}")

gravada: gold_distribuicoes


gravada: gold_cruzamentos


gravada: gold_salario


gravada: gold_mencoes
